# Common Libraries

In [1]:
import os, shutil, sys
if shutil.which('nvidia-smi') is not None:
    os.environ['MUJOCO_GL'] = 'egl'
import mujoco, mujoco.viewer

import numpy as np
from pathlib import Path
from loop_rate_limiters import RateLimiter

import mink
from mink.lie import SE3
import matplotlib.pylab as plt
from IPython.display import display, clear_output
from ipywidgets import interact, FloatSlider, HBox, VBox, interactive_output, Label

# Params

In [2]:
model_path = "/home/seojin/biomechanics_tools/mink/examples/unitree_g1/scene.xml"
solver = "daqp"

feet = ["right_foot", "left_foot"]

# Initialize model & data

In [3]:
model = mujoco.MjModel.from_xml_path(model_path)
configuration = mink.Configuration(model)
configuration.update_from_keyframe("stand")
data = configuration.data

# Mocap id
com_mid = model.body("com_target").mocapid[0]
l_feet_mid = model.body("left_foot_target").mocapid[0]
r_feet_mid = model.body("right_foot_target").mocapid[0]
r_hand_mid = model.body("right_palm_target").mocapid[0]
l_hand_mid = model.body("left_palm_target").mocapid[0]

# Initial pose
init_com = data.subtree_com[1].copy()
init_r_hand_pos = data.mocap_pos[r_hand_mid].copy()

# Render

In [4]:
renderer = mujoco.Renderer(model, height=400, width=600)
scene_option = mujoco.MjvOption()
camera = mujoco.MjvCamera()
renderer.update_scene(configuration.data, camera = camera, scene_option = scene_option)

In [5]:
def render_withCameraControl(cam_dist, cam_azim, cam_elev):
    mujoco.mj_forward(configuration.model, configuration.data)

    camera.distance = cam_dist
    camera.azimuth = cam_azim
    camera.elevation = cam_elev
    
    renderer.update_scene(configuration.data, camera=camera)
    pixels = renderer.render()

    clear_output(wait=True) 
    
    plt.figure(figsize=(10, 6))
    plt.imshow(pixels)
    plt.axis("off")
    plt.title(f"Cam: Dist={cam_dist:.1f}, Azim={cam_azim}, Elev={cam_elev}")
    plt.show()

c_dist = FloatSlider(min=1.0, max=10.0, step=0.1, value=4.0, description='Dist', continuous_update=False)
c_azim = FloatSlider(min=-180, max=180, step=1, value=180, description='Azim', continuous_update=False)
c_elev = FloatSlider(min=-90, max=90, step=1, value=-30, description='Elev', continuous_update=False)

out = interactive_output(render_withCameraControl, {
    "cam_dist": c_dist, 
    "cam_azim": c_azim, 
    "cam_elev": c_elev
})

camera_ui = VBox([Label(value="[ Camera Control ]"), c_dist, c_azim, c_elev])
ui = VBox([camera_ui, out])

display(ui)

# Tasks

In [6]:
tasks = [
    # Pelvis orientation: keep upright. No position control (floating base).
    pelvis_orientation_task := mink.FrameTask(
        frame_name="pelvis",
        frame_type="body",
        position_cost=0.0,
        orientation_cost=10.0,
        lm_damping=1e-3,
    ),

    # Torso orientation: less critical than pelvis.
    torso_orientation_task := mink.FrameTask(
        frame_name="torso_link",
        frame_type="body",
        position_cost=0.0,
        orientation_cost=5.0,
        lm_damping=1e-3,
    ),
    
    # Posture: null-space regularization only. Low cost so it never
    # interferes with higher-priority tasks.
    posture_task := mink.PostureTask(model, cost=1e-1),

    # COM: balance-critical, always feasible since we control the target.
    com_task := mink.ComTask(cost=50.0),
]

posture_task.set_target_from_configuration(configuration)
pelvis_orientation_task.set_target_from_configuration(configuration)
torso_orientation_task.set_target_from_configuration(configuration)

In [7]:
# Feet: support base, highest priority. Zero damping — targets are the
# initial positions which are always feasible.
left_foot_task = mink.FrameTask(frame_name="left_foot",
                                frame_type="site",
                                position_cost=100.0,
                                orientation_cost=10.0,
                                lm_damping=0.0)
tasks.append(left_foot_task)

right_foot_task = mink.FrameTask(frame_name="right_foot",
                                 frame_type="site",
                                 position_cost=100.0,
                                 orientation_cost=10.0,
                                 lm_damping=0.0)
tasks.append(right_foot_task)

In [8]:
hand_tasks = []

# Left hand: torso-relative circle via RelativeFrameTask.
relative_task = mink.RelativeFrameTask(
    frame_name="left_palm",
    frame_type="site",
    root_name="torso_link",
    root_type="body",
    position_cost=5.0,
    orientation_cost=0.5,
    gain=0.5,
    lm_damping=1e-3,
)
relative_task.set_target_from_configuration(configuration)
init_relative_target = relative_task.transform_target_to_root
hand_tasks.append(relative_task)

# Right hand: world-space circle via FrameTask.
r_hand_task = mink.FrameTask(
    frame_name="right_palm",
    frame_type="site",
    position_cost=5.0,
    orientation_cost=0.5,
    gain=0.5,
    lm_damping=1e-3,
)
hand_tasks.append(r_hand_task)
tasks.extend(hand_tasks)

# Limitations

In [9]:
# Collision avoidance
collision_pairs = [
    (["left_hand_collision"], ["left_thigh_collision"]),
    (["right_hand_collision"], ["right_thigh_collision"]),
]
collision_avoidance_limit = mink.CollisionAvoidanceLimit(
    model = model,
    geom_pairs = collision_pairs,  # type: ignore
    minimum_distance_from_collisions = 0.005,
    collision_detection_distance = 0.15,
)
limits = [mink.ConfigurationLimit(model), collision_avoidance_limit]

# Mocap

In [14]:
data.mocap_pos[com_mid] = data.subtree_com[1].copy()
mink.move_mocap_to_frame(model, data, "right_foot_target", "right_foot", "site")
mink.move_mocap_to_frame(model, data, "left_foot_target", "left_foot", "site")
mink.move_mocap_to_frame(model, data, "right_palm_target", "right_palm", "site")
mink.move_mocap_to_frame(model, data, "left_palm_target", "left_palm", "site")

In [15]:
model = configuration.model
data = configuration.data

# Record initial positions.
init_com = data.subtree_com[1].copy()
init_r_hand_pos = data.mocap_pos[r_hand_mid].copy()

# Set trajectory

In [16]:
squat_depth = 0.12
squat_freq = 0.5  # one squat every 2s

circle_radius = 0.04
circle_freq = 0.8

wave_radius = 0.04
wave_freq = 0.8
wave_y_offset = 0.05

dt = 1 / 500

In [17]:
n_frame = 1000
ts = np.array([(dt * frame_i) for frame_i in range(n_frame)])

# Squat
squat_phases = 0.5 * (1.0 - np.cos(2.0 * np.pi * squat_freq * ts))
com_target = init_com.copy()
com_targets = np.tile(com_target, (n_frame,1))
com_targets[:, 2] -= squat_depth * squat_phases

# Right hand: world-space circle in the sagittal plane
right_hand_angles = 2.0 * np.pi * circle_freq * ts
r_pos = init_r_hand_pos.copy()
r_poses = np.tile(r_pos, (n_frame,1))
r_poses[:, 0] = circle_radius * np.cos(right_hand_angles)
r_poses[:, 2] += circle_radius * np.sin(right_hand_angles)

# Left hand: circle in frontal plane, torso-relative
wave_ts = 2.0 * np.pi * wave_freq * ts
targets = np.zeros((n_frame, 3))
targets[:, 1] = wave_y_offset + wave_radius * np.sin(wave_ts)
targets[:, 2] = wave_radius * np.cos(wave_ts)
targets_in_root = [init_relative_target @ SE3.from_translation(t) for t in targets]

# Do inverse kinematics

In [18]:
imgs = []
for frame_i in range(n_frame):
    data.mocap_pos[com_mid] = com_targets[frame_i]
    data.mocap_pos[r_hand_mid] = r_poses[frame_i]

    # Visualize the relative task target in world space.
    torso_in_world = configuration.get_transform_frame_to_world("torso_link", "body")
    target_in_world = torso_in_world @ targets_in_root[frame_i]
    data.mocap_pos[l_hand_mid] = target_in_world.translation()
    data.mocap_quat[l_hand_mid] = target_in_world.rotation().wxyz

    # Update task targets
    com_task.set_target(data.mocap_pos[com_mid])
    
    left_foot_task.set_target(mink.SE3.from_mocap_id(data, l_feet_mid))
    right_foot_task.set_target(mink.SE3.from_mocap_id(data, r_feet_mid))
    
    r_hand_task.set_target(mink.SE3.from_mocap_id(data, r_hand_mid))
    relative_task.set_target(targets_in_root[frame_i])
    
    # Inverse kinematics
    vel = mink.solve_ik(configuration, 
                        tasks, 
                        dt, 
                        solver,
                        damping=1e-2,
                        limits=limits)
    configuration.integrate_inplace(vel, dt)

    mujoco.mj_forward(configuration.model, data) 
    mujoco.mj_camlight(configuration.model, data)

    # Images
    renderer.update_scene(data)
    imgs.append(renderer.render())
    
imgs = np.array(imgs)

# Plot

In [19]:
def view_image(n = 0):
    plt.figure(figsize=(8, 6))
    plt.imshow(imgs[n])
    plt.title(f'Image Index: {n}')
    plt.axis('off')
    plt.show()

interact(view_image, n=(0, len(imgs) - 1))

interactive(children=(IntSlider(value=0, description='n', max=999), Output()), _dom_classes=('widget-interact'…

<function __main__.view_image(n=0)>